# 04 — Training Curves
**Goal:** Master the most common figure in ML papers — loss/accuracy curves — with confidence bands, multi-run comparisons, log scaling, and key-point annotations.

By the end of this notebook you will:
- Plot training curves with shaded mean ± std bands across multiple runs
- Compare multiple models cleanly on one axes
- Apply exponential moving average (EMA) smoothing to noisy curves
- Use log scale appropriately
- Annotate key events (best checkpoint, divergence point)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('research.mplstyle')

np.random.seed(42)

epochs = np.arange(1, 101)

def simulate_runs(n_runs, decay, offset, noise):
    """Simulate n_runs training loss curves with realistic noise."""
    runs = []
    for _ in range(n_runs):
        seed_offset = np.random.randn() * 0.05   # per-run variance
        curve = (2.5 + seed_offset) * np.exp(-decay * epochs) + offset + noise * np.random.randn(100)
        runs.append(curve)
    return np.array(runs)   # shape: (n_runs, 100)

# 5 runs each for two models
model_a_runs = simulate_runs(5, decay=0.06, offset=0.15, noise=0.07)
model_b_runs = simulate_runs(5, decay=0.09, offset=0.10, noise=0.07)

---
## Part 1 — Single run curve

Start simple. Plot one run of model_a (the first row) as a line.

**Exercise:** Plot `model_a_runs[0]` vs `epochs` with proper labels.

In [ ]:
# TODO: plot a single training loss curve
# x = epochs, y = model_a_runs[0]
# Add x-label, y-label, and a legend entry


---
## Part 2 — Multi-run mean ± std with shaded band

Plotting a single run hides variance. Professional ML papers show the **mean** as a solid line and shade the **mean ± std** region across runs.

The key function: `ax.fill_between(x, lower, upper, alpha=0.2)`

**Exercise:** Compute mean and std across the 5 runs of model_a, then plot with a shaded band.

In [ ]:
# TODO: compute mean and std of model_a_runs across axis=0
# TODO: plot the mean line
# TODO: shade mean ± std using ax.fill_between()
#   Use the same color as the line but with alpha=0.15 for the fill
#   Hint: ax.fill_between(epochs, mean - std, mean + std, alpha=0.15, color=...)


---
## Part 3 — Comparing two models

Now show both models on the same axes. Each gets its own color, mean line, and shaded band.

**Exercise:** Plot model_a and model_b with mean ± std bands. Use two distinct Okabe-Ito colors.

In [ ]:
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7']

# TODO: plot model_a (blue, OKABE_ITO[4]) and model_b (orange, OKABE_ITO[0])
# Each with mean line + shaded std band
# Add legend distinguishing the two models


---
## Part 4 — EMA smoothing for noisy curves

Real training curves are often spiky. Exponential Moving Average (EMA) smoothing reveals the trend without hiding it behind noise.

EMA formula: `smoothed[t] = alpha * raw[t] + (1 - alpha) * smoothed[t-1]`

Typical alpha values: 0.1–0.3 (lower = smoother).

The pattern used by top labs: plot the **smoothed line** prominently, and the **raw curve** faintly in the background.

**Exercise:** Implement EMA, then plot raw (faint, alpha=0.3) + smoothed (solid) for one run.

In [ ]:
def ema_smooth(values, alpha=0.2):
    """Exponential moving average smoothing."""
    # TODO: implement EMA
    # Initialize smoothed[0] = values[0]
    # For each subsequent index: smoothed[i] = alpha * values[i] + (1 - alpha) * smoothed[i-1]
    # Return the smoothed array
    pass

In [ ]:
raw = model_a_runs[0]
smoothed = ema_smooth(raw, alpha=0.15)

# TODO: plot raw with alpha=0.3 and color='#56B4E9'
# TODO: plot smoothed with alpha=1.0, same color, slightly thicker line
# Add a legend distinguishing 'Raw' vs 'Smoothed'


---
## Part 5 — Log scale

Loss that drops orders of magnitude (common in LLM pretraining) is best shown on a log y-axis. This makes early and late training equally readable.

**Exercise:** Simulate a loss that drops from 10 to 0.01, and compare linear vs. log y-axis in a side-by-side plot.

In [ ]:
steps     = np.arange(1, 1001)
llm_loss  = 10 * np.exp(-0.008 * steps) + 0.01 + 0.05 * np.random.randn(1000)
llm_loss  = np.clip(llm_loss, 0.01, None)

# TODO: create a 1×2 figure
# Left: linear y-axis
# Right: log y-axis using ax.set_yscale('log')
# Label both and add a title to each ("Linear" and "Log scale")


---
## Part 6 — Annotating key events

Papers often annotate figures: mark the best checkpoint, a learning rate drop, or the point where one model overtakes another.

Useful tools:
- `ax.axvline(x, ...)` — vertical reference line
- `ax.annotate(text, xy=..., xytext=..., arrowprops=...)` — arrow + label

**Exercise:** On the model_a vs model_b comparison (Part 3), annotate:
1. A vertical dashed line at the epoch where model_b's mean loss first drops below model_a's
2. A text label for that event

In [ ]:
mean_a = model_a_runs.mean(axis=0)
mean_b = model_b_runs.mean(axis=0)

# TODO: find the first epoch where mean_b < mean_a
# Hint: np.argmax(...) or np.where(...)[0][0]

# TODO: reproduce the Part 3 comparison plot
# Add ax.axvline() at the crossover epoch (dashed, gray)
# Add ax.annotate() with text 'Model B takes lead' and an arrow pointing to the crossover
# Hint for annotate:
#   ax.annotate('text', xy=(x_point, y_point), xytext=(x_text, y_text),
#               arrowprops=dict(arrowstyle='->', color='gray'),
#               fontsize=9, color='gray')


---
## Summary

| Technique | Code pattern |
|-----------|-------------|
| Mean ± std band | `ax.fill_between(x, mean-std, mean+std, alpha=0.15)` |
| EMA smoothing | Custom loop: `s[i] = α·x[i] + (1-α)·s[i-1]` |
| Log scale | `ax.set_yscale('log')` |
| Vertical reference | `ax.axvline(x, linestyle='--', color='gray')` |
| Arrow annotation | `ax.annotate(text, xy=..., xytext=..., arrowprops=...)` |

**Next:** `05_comparisons.ipynb` — bar charts and grouped comparisons for model benchmarks.